# 02 — Generate synthetic call transcripts

This notebook generates `sample_size` synthetic call transcripts that mimic what a fraud-detection tool would process. The same transcripts are used by both AI Functions paths (`ai_extract` and `ai_query`) so the cost comparison is apples-to-apples.

**What this notebook does:**
1. Reads job parameters.
2. Builds `sample_size` transcripts with controlled fraud-signal injection (~30% of rows have a fraud phrase, the rest are benign).
3. Writes them to `<catalog>.<schema>.transcripts` as a Delta table.
4. Prints distribution stats so you can verify the mix.

**Why a fixed seed:** re-runs produce identical input data — so cost predictions and reconciliation are reproducible.

**Public references:**
- [Delta Lake basics](https://docs.databricks.com/en/delta/index.html)
- [`spark.createDataFrame`](https://api-docs.databricks.com/python/pyspark/latest/pyspark.sql/api/pyspark.sql.SparkSession.createDataFrame.html)


## Read job parameters and resolve table names

Same widget pattern as `01_setup`. Every table name is built from `<catalog>.<schema>` so nothing is hardcoded.


In [ ]:
dbutils.widgets.text("catalog",      "main")
dbutils.widgets.text("schema",       "ai_functions_cost_demo")
dbutils.widgets.text("sample_size",  "100")
dbutils.widgets.text("team_tag",     "ai-cost-demo")
dbutils.widgets.text("owner_email",  "you@example.com")

CATALOG     = dbutils.widgets.get("catalog")
SCHEMA      = dbutils.widgets.get("schema")
SAMPLE_SIZE = int(dbutils.widgets.get("sample_size"))

TRANSCRIPTS_TABLE = f"{CATALOG}.{SCHEMA}.transcripts"
print(f"Will write to: {TRANSCRIPTS_TABLE}  (sample_size={SAMPLE_SIZE})")


## Generate the transcripts

The next cell builds the transcripts in pure Python (deterministic via `random.seed(42)`), then converts to a Spark DataFrame and writes Delta with `overwrite + overwriteSchema` so re-runs are clean.

**Schema of each transcript row:**
- `transcript_id` — stable id like `TX-00001`
- `caller_name_truth`, `account_id_truth`, `dollar_amount_truth`, `is_fraud_truth`, `fraud_phrase_truth` — ground-truth fields the AI calls will be scored against in the cost-reconciliation step
- `transcript` — the conversational text body (~600 chars)

**Fraud signal injection:** ~30% of rows contain one of six realistic fraud phrases (`"I did not authorize this transaction"`, `"my card was stolen"`, etc.). The other ~70% have benign reasons (address change, duplicate charge, etc.).


In [ ]:
import random
from pyspark.sql import Row

random.seed(42)

CALLER_NAMES = ["Maria Garcia", "James Lee", "Emma Patel", "Liam Smith",
                "Ava Johnson", "Noah Brown", "Sophia Davis", "Lucas Wilson",
                "Mia Anderson", "Ethan Martinez"]
ACCOUNT_PREFIXES = ["CHK", "SAV", "CC", "LOAN"]
FRAUD_PHRASES = [
    "I did not authorize this transaction",
    "this charge is not me",
    "my card was stolen yesterday",
    "someone is using my account",
    "I never bought anything at that store",
    "my login credentials were phished last week",
]
BENIGN_REASONS = [
    "I want to update my address",
    "I would like to dispute a duplicate charge",
    "I need to set up direct deposit",
    "checking my statement balance",
    "asking about an upcoming payment",
]

def gen(i):
    caller   = random.choice(CALLER_NAMES)
    account  = f"{random.choice(ACCOUNT_PREFIXES)}-{random.randint(100000, 999999)}"
    amount   = round(random.uniform(15.00, 4250.00), 2)
    is_fraud = random.random() < 0.30
    phrase   = random.choice(FRAUD_PHRASES) if is_fraud else None
    reason   = phrase if is_fraud else random.choice(BENIGN_REASONS)
    transcript = (
        f"Agent: Thank you for calling First National Bank, my name is Sarah. How can I help?\n"
        f"Customer: Hi Sarah, this is {caller}. I'm calling about my account {account}.\n"
        f"Agent: Of course, can you verify the last four digits of the account?\n"
        f"Customer: Yes, it ends in {account[-4:]}.\n"
        f"Agent: Great, I see the account. What can I help with?\n"
        f"Customer: {reason}. There's a charge for ${amount:.2f} that I'm asking about.\n"
        f"Agent: I see the ${amount:.2f} charge here. Let me look into that for you.\n"
        f"Customer: Thanks, please look into the ${amount:.2f} on account {account}.\n"
    )
    return {
        "transcript_id":       f"TX-{i:05d}",
        "caller_name_truth":   caller,
        "account_id_truth":    account,
        "dollar_amount_truth": amount,
        "is_fraud_truth":      is_fraud,
        "fraud_phrase_truth":  phrase,
        "transcript":          transcript,
    }

rows = [gen(i) for i in range(SAMPLE_SIZE)]
df = spark.createDataFrame([Row(**r) for r in rows])
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TRANSCRIPTS_TABLE)

n_fraud  = sum(1 for r in rows if r["is_fraud_truth"])
avg_chars = sum(len(r["transcript"]) for r in rows) // len(rows)

print(f"Wrote {len(rows):,} transcripts to {TRANSCRIPTS_TABLE}")
print(f"  Fraud rows:   {n_fraud:,} ({100*n_fraud//len(rows)}%)")
print(f"  Avg chars:    {avg_chars:,}")
display(spark.table(TRANSCRIPTS_TABLE).limit(3))


## Done

The transcripts table is ready. Both downstream tasks (`03a_run_ai_extract` and `03b_run_ai_query`) read from this same table, so the cost comparison is apples-to-apples.
